# Functions and classes

## Models for benchmark data

In [ ]:
from uuid import UUID
from datetime import datetime
from enum import Enum
from pydantic import BaseModel


class Roles(Enum):
    TestingAgent = 'testing_agent'
    TestedAgent = 'tested_agent'


class MetricType(Enum):
    Binary = 'Binary'
    Numeric = 'Numeric'


class Message(BaseModel):
    role: str
    content: str | None = None
    time_start: float
    time_end: float
    timestamp: datetime | None = None
    interrupted: bool = False


class Transcript(BaseModel):
    messages: list[Message]


class CallRecording(BaseModel):
    id: UUID
    simulation_id: UUID
    test_case: str
    scenario: str
    persona: str
    transcript: Transcript
    audio_path: str
    expected_outcome: str


class Metric(BaseModel):
    id: UUID
    name: str
    description: str
    type: MetricType
    prompt: str


class Result(BaseModel):
    id: UUID
    call_recording_id: UUID
    metric_id: UUID
    contestant: str
    type: MetricType
    value: float


class EvaluationBenchmark(BaseModel):
    call_recordings: dict[UUID, CallRecording]
    metrics: dict[UUID, Metric]
    results: dict[UUID, Result]

## Models for Prolific study data and responses

In [ ]:
from typing import Literal, Annotated
from uuid import UUID
from datetime import datetime

from pydantic import BaseModel, Field, ConfigDict
from pydantic.alias_generators import to_camel


class FileData(BaseModel):
    uuid: str
    survey_type: Literal['comparison', 'evaluation']
    title: str
    header_description: str | None


class Message(BaseModel):
    role: Literal['user', 'assistant']
    content: str
    start_time: float
    end_time: float


class Recording(BaseModel):
    id: str
    title: str
    transcript: list[Message]
    audio_url: str | None


class Question(BaseModel):
    id: str
    prompt: str
    type: str


class BinaryQuestion(Question):
    type: Literal['binary'] = 'binary'
    options: list[str] = Field(default_factory=lambda: ['yes', 'no'])  # noqa


class ComparisonQuestion(Question):
    type: Literal['comparison'] = 'comparison'
    options: list[str] = Field(default_factory=lambda: ['left', 'equal', 'right'])  # noqa


class Range(BaseModel):
    min: float
    max: float


class NumericQuestion(Question):
    type: Literal['numeric'] = 'numeric'
    range: Range


QuestionType = Annotated[
    BinaryQuestion | ComparisonQuestion | NumericQuestion,
    Field(discriminator='type')
]


class SurveyData(BaseModel):
    file_data: FileData
    recordings: list[Recording]
    questions: list[QuestionType]


class ResponseBaseModel(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel,
        populate_by_name=True,
        from_attributes=True,
    )


class Response(ResponseBaseModel):
    question_id: str
    response: str | int
    feedback: str


class AudioAnalytics(ResponseBaseModel):
    total_audio_duration: float
    total_listening_time: float
    overall_coverage: float
    audio_files: list[dict]


class SurveyMetadata(ResponseBaseModel):
    title: str
    scenario: str | None = None
    total_questions: int
    audio_files: int
    transcripts: int


class ResponseData(ResponseBaseModel):
    prolific_id: str
    page_id: str
    uuid: UUID
    survey_type: str
    responses: list[Response]
    time_taken: float
    audio_listening_analytics: AudioAnalytics
    survey: SurveyMetadata

## Other classes and utils

In [ ]:
def make_metric_name(metric: Metric) -> str:
    return f'{metric.name} ({str(metric.id)[:4]})'

In [ ]:
import numpy as np
from dataclasses import dataclass


@dataclass
class ResponseValue:
    response: str
    respondent_id: str


def aggregate_binary_responses(responses: list[ResponseValue]) -> float:
    values = [r.response.lower() == 'yes' for r in responses]
    consensus = np.mean(values) > 0.5
    consensus_count = sum([v == consensus for v in values])
    return float(int(consensus)), consensus_count

def aggregate_numeric_responses(responses: list[ResponseValue]) -> float:
    values = [float(r.response) for r in responses]
    return float(np.median(values))

## Evaluation functions

In [ ]:
from uuid import UUID

from sqlalchemy import select
from sqlalchemy.orm import Session
import pandas as pd


def load_eval_results(
        benchmark: EvaluationBenchmark,
        contestant: str,
) -> pd.DataFrame:
        
    result_by_recording_and_metric: dict[tuple[UUID, UUID], Result] = {}
    for result_id, result in benchmark.results.items():
        if result.contestant != contestant:
            continue
        
        key = (result.call_recording_id, result.metric_id)

        if key in result_by_recording_and_metric:
            print(f'WARNING: duplicated result with ID "{result_id}"')
            continue

        result_by_recording_and_metric[key] = result

    rows = []
    for call_recording_id, call_recording in benchmark.call_recordings.items():
        row = {
            'call_recording_id': call_recording_id
        }

        for metric_id, metric in benchmark.metrics.items():
            result = result_by_recording_and_metric[(call_recording_id, metric_id)]
            
            row[make_metric_name(metric)] = result.value

        rows.append(row)

    return pd.DataFrame(rows).set_index('call_recording_id')

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, mean_absolute_error, mean_squared_error
import numpy as np


def evaluate_contestant_performance(
        contastant_df: pd.DataFrame,
        ground_truth_df: pd.DataFrame,
        binary_metric_names: list[str],
        numeric_metric_names: list[str],
):
    # Ensure same records in both datasets
    common_ids = contastant_df.index.intersection(ground_truth_df.index)
    contastant_df = contastant_df.loc[common_ids]
    ground_truth_df = ground_truth_df.loc[common_ids]

    # Binary metrics evaluation
    binary_scores = []
    for metric in binary_metric_names:
        if metric not in contastant_df.columns or metric not in ground_truth_df.columns:
            continue
            
        y_true = ground_truth_df[metric].astype(int)
        y_pred = contastant_df[metric].astype(int)
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='binary', zero_division=0
        )
        
        binary_scores.append({
            'metric': metric,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'accuracy': (y_true == y_pred).mean()
        })

    # Numeric metrics evaluation
    numeric_scores = []
    for metric in numeric_metric_names:
        if metric not in contastant_df.columns or metric not in ground_truth_df.columns:
            continue
            
        y_true = ground_truth_df[metric]
        y_pred = contastant_df[metric]
        
        numeric_scores.append({
            'metric': metric,
            'mae': mean_absolute_error(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'correlation': y_true.corr(y_pred)
        })

    binary_scores_df = pd.DataFrame(binary_scores).set_index('metric') if len(binary_scores) > 0 else None
    numeric_scores_df = pd.DataFrame(numeric_scores).set_index('metric') if len(numeric_scores) > 0 else None

    return binary_scores_df, numeric_scores_df

## Visualization functions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def visualize_binary_scores(binary_scores_by_contestant: dict[str: pd.DataFrame]):
    # Combine all DataFrames
    combined_data = []
    for contestant_name, df in binary_scores_by_contestant.items():
        df_copy = df.copy()
        df_copy['Contestant'] = contestant_name
        df_copy['metric'] = df_copy.index
        combined_data.append(df_copy)
    
    combined_df = pd.concat(combined_data, ignore_index=True)
    
    # Create subplots for each score type
    fig, axes = plt.subplots(1, 4, figsize=(18, 6))
    score_types = ['precision', 'recall', 'f1_score', 'accuracy']
    score_labels = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
    
    for i, (score_type, label) in enumerate(zip(score_types, score_labels)):
        pivot_data = combined_df.pivot(index='metric', columns='Contestant', values=score_type)
        
        ax = pivot_data.plot(kind='bar', ax=axes[i], width=0.8)
        ax.set_title(f'{label} Comparison', fontsize=14, fontweight='bold')
        ax.set_xlabel('', fontsize=12)
        ax.set_ylabel(label, fontsize=12)
        # ax.legend(title='Contestant', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.legend(loc='lower left')
        ax.tick_params(axis='x', rotation=90)
        ax.set_ylim(0, 1.2)
        
        # Add value labels on bars
        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', rotation=90, padding=3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
from typing import Literal
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


def draw_radar_chart_for_binary_scores(
        binary_scores_by_contestant: dict[str: pd.DataFrame],
        metric_name: Literal['precision', 'recall', 'f1_score', 'accuracy'] = 'f1_score'
):    
    # Get all unique metrics
    all_binary_metrics = set()
    
    for df in binary_scores_by_contestant.values():
        all_binary_metrics.update(df.index)
    
    # Prepare categories
    categories = []
    for metric in sorted(all_binary_metrics):
        categories.append(f'{metric.split(" (")[0]}\n(F1)')
    
    # Prepare data for each contestant
    radar_data = {}
    for contestant_name in binary_scores_by_contestant.keys():
        scores = []
        
        # Binary metrics (F1 scores)
        for metric in sorted(all_binary_metrics):
            scores.append(binary_scores_by_contestant[contestant_name].loc[metric, metric_name])
        
        radar_data[contestant_name] = scores
    
    # Create radar chart
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))  # Complete the circle
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728', '#9467bd']
    for i, (contestant_name, scores) in enumerate(radar_data.items()):
        values = scores + [scores[0]]  # Complete the circle
        ax.plot(angles, values, 'o-', linewidth=2, label=contestant_name, color=colors[i % len(colors)])
        ax.fill(angles, values, alpha=0.25, color=colors[i % len(colors)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_ylabel('', fontsize=12)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    plt.title('Radar chart for binary metrics', fontsize=16, fontweight='bold', pad=30)
    plt.tight_layout()
    plt.show()

In [ ]:
def visualize_numeric_scores(numeric_scores_by_contestant: dict[str: pd.DataFrame]):        
    # Combine all numeric DataFrames
    combined_data = []
    for contestant_name, df in numeric_scores_by_contestant.items():
        df_copy = df.copy()
        df_copy['Contestant'] = contestant_name
        df_copy['metric'] = df_copy.index
        combined_data.append(df_copy)
    
    combined_df = pd.concat(combined_data, ignore_index=True)
    
    # Create subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    # MAE comparison (lower is better)
    pivot_mae = combined_df.pivot(index='metric', columns='Contestant', values='mae')
    pivot_mae.plot(kind='bar', ax=axes[0], color=colors)
    axes[0].set_title('Mean Absolute Error (Lower is Better)', fontweight='bold')
    axes[0].set_ylabel('MAE')
    axes[0].set_ylim(0, 1.5)
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(title='Contestants')
    
    # Add value labels
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt='%.2f', rotation=90, padding=3)
    
    # RMSE comparison (lower is better)
    pivot_rmse = combined_df.pivot(index='metric', columns='Contestant', values='rmse')
    pivot_rmse.plot(kind='bar', ax=axes[1], color=colors)
    axes[1].set_title('Root Mean Square Error (Lower is Better)', fontweight='bold')
    axes[1].set_ylabel('RMSE')
    axes[1].set_ylim(0, 1.9)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(title='Contestants')
    
    # Add value labels
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt='%.2f', rotation=90, padding=3)
    
    # Correlation comparison (higher is better)
    pivot_corr = combined_df.pivot(index='metric', columns='Contestant', values='correlation')
    pivot_corr.plot(kind='bar', ax=axes[2], color=colors)
    axes[2].set_title('Correlation (Higher is Better)', fontweight='bold')
    axes[2].set_ylabel('Correlation')
    axes[2].set_ylim(0, 0.9)
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].legend(title='Contestants')
    
    # Add value labels
    for container in axes[2].containers:
        axes[2].bar_label(container, fmt='%.2f', rotation=90, padding=3)
    
    plt.tight_layout()
    plt.show()

# Computation and visualization of the scores

In [ ]:
evaluation_data_path = 'study-eval-0923/eval-data.json'
responses_path = 'study-eval-0923/responses'

use_subset_of_call_recordings = False  # set to True to use a subset of recordings for testing purposes

In [ ]:
# Call recordings with consensus >=80%
subset_of_call_ids = [
	UUID('2bcaf463-9c2d-49e2-bff2-36a298d125d7'),
	UUID('38af01b8-3319-4855-a048-207509531065'),
	UUID('8e89f29c-dd6a-415a-adb6-670684ae79af'),
	UUID('b6005001-09ae-4670-8d7d-a42e66500a9e'),
	UUID('97d76263-22cc-46ad-9ee1-18d8cc64abfc'),
	UUID('7f31b48a-6543-4389-852e-db155c8d22ca'),
	UUID('aef93e06-88e1-4e60-8201-d04de9e25f90'),
	UUID('3644e54a-4de4-4c1d-8946-a831695021de'),
	UUID('8f4d6f11-af38-4fb1-bec8-6167907f1dfb'),
	UUID('7d7d2c80-2828-479c-93f7-6d515fbf99c5'),
	UUID('08d4e720-c31d-4dc9-b787-e54b687a0e87'),
	UUID('25a34317-0165-485d-a41f-39c146f601fe'),
	UUID('9cd0a87a-7d8c-4829-aa3a-a03b54915e1e'),
	UUID('f509b042-7072-425e-9466-9a482c86fc18'),
	UUID('a1a0d080-57cb-46a3-8661-e0438219759d'),
	UUID('a4250b1c-dff8-41b5-8c2e-1f9cb9ba211c'),
	UUID('77cc7b0f-a7e3-406e-a0bc-bddad312c4ea'),
	UUID('b0385cb3-993c-4bf2-9491-fa3f35423a81'),
	UUID('2c8fa34c-7743-42dd-b7bc-b809484b3f37'),
	UUID('978c04e1-fe71-439c-a106-2084ba80ba2b'),
	UUID('9f7de6de-3765-4e58-a77a-d70621007952'),
	UUID('963b9a15-c3aa-45c4-a14b-c03754c6060a'),
	UUID('7deda16b-82fc-4e2f-9086-62788e513c40'),
	UUID('3a76500d-51d3-4c1f-ac2a-e0183ef08f91'),
	UUID('a4dfaa08-0765-4cc9-918d-8614c21acb2c'),
	UUID('51283018-c98d-446f-8483-bb78939ccc4a'),
	UUID('d739f29d-ea10-496d-9a41-bab55d06bf52'),
	UUID('96eed521-cb29-4b8a-a7c8-98bf11034428'),
	UUID('403ee2bf-4a22-4aa3-a08c-27e9323e054d'),
	UUID('c57b1954-ca42-417d-b0dd-1b2626845c48'),
	UUID('33531f08-ad2c-4d57-ac91-9ee56493afa6'),
	UUID('8779f274-9132-49d4-ba95-1b14c2c87bea'),
	UUID('4a81dc7d-e3d0-47b5-b088-11063cf674f5'),
	UUID('da452a30-7ef1-44b7-abe2-19858c5dad50'),
	UUID('0c5a88cf-702d-45ea-8df3-33a9db8f0270'),
	UUID('3e3119bc-f709-4833-89b3-05250163252c'),
	UUID('d32d8f29-b5cb-4870-bad0-e9ecd3e1a4d6'),
	UUID('298f50c4-cbc9-40ef-aa7e-26686efa181c'),
	UUID('4056c2d1-3695-4ee5-b168-3e659f2efd86'),
	UUID('6dc0306a-28ce-458f-b9d8-720393982f61'),
	UUID('b076ce25-8900-4b11-b081-9ee3da792ede'),
	UUID('16c6b506-b0b5-4f6c-b52e-9d11cd861b95'),
	UUID('190f0a28-5b4b-4797-b188-f9c16691ec2a'),
	UUID('c1e7533e-0667-43ca-8453-bc7d02447a2a'),
	UUID('a321dba8-ee02-499c-afb0-5ce386eacb5b'),
]

In [ ]:
import json


with open(evaluation_data_path, 'r') as f:
    data = json.loads(f.read())

benchmark = EvaluationBenchmark.model_validate(data)

binary_metric_names = [make_metric_name(m) for m_id, m in benchmark.metrics.items() if m.type == MetricType.Binary]
numeric_metric_names = [make_metric_name(m) for m_id, m in benchmark.metrics.items() if m.type == MetricType.Numeric]

## Aggregate responses to get ground truth labels

In [ ]:
from os import path as osp, listdir
import json


response_file_paths = []
for item in listdir(responses_path):
    filename, ext = osp.splitext(item)
    if ext != '.json':
        continue

    response_file_paths.append(osp.join(responses_path, item))

study_results = []
for file_path in response_file_paths:
    with open(file_path, 'r') as f:
        data = json.loads(f.read())
    response_data = ResponseData.model_validate(data)
    study_results.append(response_data)

In [ ]:
from uuid import UUID
from collections import Counter


# Collect all responses, grouped by (call_recording_id, metric_id) 
responses_by_recording_and_metric: dict[tuple[UUID, UUID], list[ResponseValue]] = {}
call_recording_ids = set()
metric_ids = set()
for study_result in study_results:
    call_recording_id = study_result.uuid
    call_recording_ids.add(call_recording_id)
    for response in study_result.responses:
        metric_id = UUID(response.question_id)
        metric_ids.add(metric_id)

        key = (call_recording_id, metric_id)
        responses = responses_by_recording_and_metric.get(key, [])
        responses.append(ResponseValue(
            response=response.response,
            respondent_id=study_result.prolific_id,
        ))
        responses_by_recording_and_metric[key] = responses

print(f'In {len(study_results)} responses there are:')
print(f'  * {len(call_recording_ids)} distinct call recording IDs')
print(f'  * {len(metric_ids)} distinct metric IDs')
print()

# Check for duplicated responses from the same prolific ID
max_num_responses = 0
min_num_responses = len(study_results)
for (call_recording_id, metric_id), responses in responses_by_recording_and_metric.items():
    max_num_responses = max(max_num_responses, len(responses))
    min_num_responses = min(min_num_responses, len(responses))
    counts = Counter([r.respondent_id for r in responses])
    for respondent_id, count in counts.items():
        if count > 1:
            print(f'WARNING: found {count} responses from respondent with ID "{respondent_id}" for call_recording_id={call_recording_id} and metric_id={metric_id}')

print(f'Max number of responses: {max_num_responses}')
print(f'Min number of responses: {min_num_responses}')

In [ ]:
import pandas as pd


aggregated_results = []
call_recording_ids = subset_of_call_ids if use_subset_of_call_recordings else list(benchmark.call_recordings.keys())
for call_recording_id in call_recording_ids:
    result = {
        'call_recording_id': call_recording_id
    }
    
    for metric_id, metric in benchmark.metrics.items():
        responses = responses_by_recording_and_metric[(call_recording_id, metric_id)]
        assert len(responses) > 0

        if metric.type == MetricType.Binary:
            aggregated_value, consensus_count = aggregate_binary_responses(responses)
        elif metric.type == MetricType.Numeric:
            aggregated_value = aggregate_numeric_responses(responses)
            consensus_count = 0
        else:
            raise ValueError(f'Unexpected metric type "{metric.type.value}"')

        result[make_metric_name(metric)] = aggregated_value

    aggregated_results.append(result)

ground_truth_df=pd.DataFrame(aggregated_results).set_index('call_recording_id')
len(ground_truth_df)

## Get evaluation results

In [ ]:
"""Load Evalion results """
evalion_results_df = load_eval_results(
    benchmark=benchmark,
    contestant='Evalion'
)

In [ ]:
evalion_binary_scores_df, evalion_numeric_scores_df = evaluate_contestant_performance(
    contastant_df=evalion_results_df,
    ground_truth_df=ground_truth_df,
    binary_metric_names=binary_metric_names,
    numeric_metric_names=numeric_metric_names
)

In [ ]:
# Evalion
pd.concat([evalion_binary_scores_df, evalion_numeric_scores_df])

In [ ]:
"""Load Cekura results """
cekura_results_df = load_eval_results(
    benchmark=benchmark,
    contestant='Cekura'
)

In [ ]:
cekura_binary_scores_df, cekura_numeric_scores_df = evaluate_contestant_performance(
    contastant_df=cekura_results_df,
    ground_truth_df=ground_truth_df,
    binary_metric_names=binary_metric_names,
    numeric_metric_names=numeric_metric_names
)

In [ ]:
# Cekura
pd.concat([cekura_binary_scores_df, cekura_numeric_scores_df])

In [ ]:
"""Load Coval results """
coval_results_df = load_eval_results(
    benchmark=benchmark,
    contestant='Coval'
)

In [ ]:
coval_binary_scores_df, coval_numeric_scores_df = evaluate_contestant_performance(
    contastant_df=coval_results_df,
    ground_truth_df=ground_truth_df,
    binary_metric_names=binary_metric_names,
    numeric_metric_names=numeric_metric_names
)

In [ ]:
# Coval
pd.concat([coval_binary_scores_df, coval_numeric_scores_df])

## Visualize

In [ ]:
binary_results = {
    'Evalion': evalion_binary_scores_df,
    'Cekura': cekura_binary_scores_df, 
    'Coval': coval_binary_scores_df
}

numeric_results = {
    'Evalion': evalion_numeric_scores_df,
    'Cekura': cekura_numeric_scores_df,
    'Coval': coval_numeric_scores_df
}

In [ ]:
visualize_binary_scores(binary_results)

In [ ]:
draw_radar_chart_for_binary_scores(binary_results, metric_name='f1_score')

In [ ]:
visualize_numeric_scores(numeric_results)

## Statistical Analysis - BASED ON BINARY OBSERVATIONS

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import mcnemar, cochrans_q
import warnings
warnings.filterwarnings('ignore')

class EvaluationStatisticalAnalysis:
    """
    Statistical analysis for comparing voice AI evaluation platforms.
    Uses actual recording-level binary outcomes for proper statistical testing.
    """
    
    def __init__(self, 
                 evalion_df: pd.DataFrame,
                 cekura_df: pd.DataFrame,  
                 coval_df: pd.DataFrame,
                 ground_truth_df: pd.DataFrame,
                 binary_metrics: list[str],
                 numeric_metrics: list[str]):
        """
        Initialize with evaluation results DataFrames.
        Each DataFrame should have recordings as rows and metrics as columns.
        """
        self.evalion_df = evalion_df
        self.cekura_df = cekura_df
        self.coval_df = coval_df
        self.ground_truth_df = ground_truth_df
        self.binary_metrics = binary_metrics
        self.numeric_metrics = numeric_metrics
        
        # Ensure all DataFrames have same index - MUST include ground_truth_df in intersection
        common_idx = (evalion_df.index.intersection(cekura_df.index)
                     .intersection(coval_df.index)
                     .intersection(ground_truth_df.index))

        print(f"Index intersection: {len(evalion_df)} evalion, {len(cekura_df)} cekura, "
              f"{len(coval_df)} coval, {len(ground_truth_df)} ground_truth -> {len(common_idx)} common")

        self.evalion_df = evalion_df.loc[common_idx]
        self.cekura_df = cekura_df.loc[common_idx]
        self.coval_df = coval_df.loc[common_idx]
        self.ground_truth_df = ground_truth_df.loc[common_idx]
        
        self.n_recordings = len(common_idx)
        self.n_binary_metrics = len(binary_metrics)
        
    def prepare_binary_outcomes(self):
        """
        Prepare binary correctness data for each platform.
        Returns dict with platform names as keys and arrays of 0/1 outcomes.
        """
        binary_outcomes = {}
        
        for platform_name, platform_df in [('Evalion', self.evalion_df),
                                           ('Cekura', self.cekura_df),
                                           ('Coval', self.coval_df)]:
            outcomes = []
            for metric in self.binary_metrics:
                # Compare platform predictions to ground truth
                platform_preds = platform_df[metric].astype(int)
                ground_truth = self.ground_truth_df[metric].astype(int)
                
                # 1 if correct, 0 if incorrect
                correct = (platform_preds == ground_truth).astype(int)
                outcomes.extend(correct.values)
            
            binary_outcomes[platform_name] = np.array(outcomes)
        
        return binary_outcomes
    
    def cochrans_q_test(self):
        """
        Perform Cochran's Q test for comparing multiple related binary samples.
        This is appropriate for comparing >2 platforms on paired binary data.
        """
        # Create matrix where rows are recording-metric pairs, columns are platforms
        n_total = self.n_recordings * self.n_binary_metrics
        data_matrix = np.zeros((n_total, 3))
        
        idx = 0
        for metric in self.binary_metrics:
            for recording_id in self.ground_truth_df.index:
                ground_truth_val = self.ground_truth_df.loc[recording_id, metric]
                
                data_matrix[idx, 0] = int(self.evalion_df.loc[recording_id, metric] == ground_truth_val)
                data_matrix[idx, 1] = int(self.cekura_df.loc[recording_id, metric] == ground_truth_val)
                data_matrix[idx, 2] = int(self.coval_df.loc[recording_id, metric] == ground_truth_val)
                idx += 1
        
        # Cochran's Q test - returns a _Bunch object with attributes
        result = cochrans_q(data_matrix)
        
        return {
            'test': "Cochran's Q",
            'statistic': result.statistic,
            'p_value': result.pvalue,
            'df': result.df,
            'interpretation': 'Tests if all platforms have equal accuracy on paired binary data'
        }
    
    def mcnemar_pairwise_tests(self):
        """
        Perform McNemar's test for pairwise comparisons using the actual mcnemar function.
        Appropriate for paired binary data (same recordings evaluated by different platforms).
        """
        results = []
        alpha = 0.05
        n_comparisons = 3
        bonferroni_alpha = alpha / n_comparisons
        
        platform_pairs = [
            ('Evalion', self.evalion_df, 'Cekura', self.cekura_df),
            ('Evalion', self.evalion_df, 'Coval', self.coval_df),
            ('Cekura', self.cekura_df, 'Coval', self.coval_df)
        ]
        
        for p1_name, p1_df, p2_name, p2_df in platform_pairs:
            # Count outcomes for all recording-metric pairs
            both_correct = 0
            p1_only_correct = 0
            p2_only_correct = 0
            both_incorrect = 0
            
            for metric in self.binary_metrics:
                p1_preds = p1_df[metric].astype(int)
                p2_preds = p2_df[metric].astype(int)
                ground_truth = self.ground_truth_df[metric].astype(int)
                
                p1_correct = (p1_preds == ground_truth)
                p2_correct = (p2_preds == ground_truth)
                
                both_correct += ((p1_correct) & (p2_correct)).sum()
                p1_only_correct += ((p1_correct) & (~p2_correct)).sum()
                p2_only_correct += ((~p1_correct) & (p2_correct)).sum()
                both_incorrect += ((~p1_correct) & (~p2_correct)).sum()
            
            # Create McNemar's contingency table in the correct format
            # Format: [[both_correct, p1_only_correct], [p2_only_correct, both_incorrect]]
            contingency_table = np.array([[both_correct, p1_only_correct],
                                         [p2_only_correct, both_incorrect]])
            
            # Use the actual McNemar function from statsmodels
            try:
                mcnemar_result = mcnemar(contingency_table, exact=True)
                p_value = mcnemar_result.pvalue
                statistic = mcnemar_result.statistic
                test_type = 'exact (statsmodels)'
            except:
                # Fallback to chi-square if exact test fails
                mcnemar_result = mcnemar(contingency_table, exact=False)
                p_value = mcnemar_result.pvalue  
                statistic = mcnemar_result.statistic
                test_type = 'chi-square (statsmodels)'
            
            # Calculate agreement and effect size
            total = both_correct + p1_only_correct + p2_only_correct + both_incorrect
            agreement = (both_correct + both_incorrect) / total
            
            # Odds ratio as effect size for McNemar
            if p2_only_correct > 0:
                odds_ratio = p1_only_correct / p2_only_correct
            else:
                odds_ratio = np.inf if p1_only_correct > 0 else 1.0
            
            # Cohen's kappa for agreement
            p_o = agreement  # Observed agreement
            p1_yes = (both_correct + p1_only_correct) / total
            p2_yes = (both_correct + p2_only_correct) / total
            p_e = p1_yes * p2_yes + (1-p1_yes) * (1-p2_yes)  # Expected agreement
            
            if p_e < 1:
                kappa = (p_o - p_e) / (1 - p_e)
            else:
                kappa = 1.0
            
            results.append({
                'comparison': f"{p1_name} vs {p2_name}",
                'test_type': f"McNemar's ({test_type})",
                'statistic': statistic,
                'p_value': p_value,
                'p_adjusted': min(p_value * n_comparisons, 1.0),
                'significant': p_value * n_comparisons < alpha,
                'agreement': agreement,
                'cohens_kappa': kappa,
                'odds_ratio': odds_ratio,
                'contingency_table': {
                    'both_correct': both_correct,
                    f'{p1_name}_only': p1_only_correct,
                    f'{p2_name}_only': p2_only_correct,
                    'both_incorrect': both_incorrect
                },
                'bonferroni_threshold': bonferroni_alpha
            })
        
        return results
    
    def calculate_effect_sizes(self):
        """
        Calculate various effect sizes for platform comparisons.
        """
        binary_outcomes = self.prepare_binary_outcomes()
        effect_sizes = []
        
        comparisons = [
            ('Evalion', 'Cekura'),
            ('Evalion', 'Coval'),
            ('Cekura', 'Coval')
        ]
        
        for p1, p2 in comparisons:
            data1 = binary_outcomes[p1]
            data2 = binary_outcomes[p2]
            
            # Cohen's h for difference in proportions
            prop1 = np.mean(data1)
            prop2 = np.mean(data2)
            h = 2 * (np.arcsin(np.sqrt(prop1)) - np.arcsin(np.sqrt(prop2)))
            
            # Cliff's Delta (non-parametric effect size)
            n1, n2 = len(data1), len(data2)
            greater = sum([(x > y) for x in data1 for y in data2])
            less = sum([(x < y) for x in data1 for y in data2])
            cliff_delta = (greater - less) / (n1 * n2)
            
            # Interpret effect sizes
            h_interp = self._interpret_cohens_h(abs(h))
            cliff_interp = self._interpret_cliff_delta(abs(cliff_delta))
            
            effect_sizes.append({
                'comparison': f"{p1} vs {p2}",
                'prop_diff': prop1 - prop2,
                'cohens_h': h,
                'h_interpretation': h_interp,
                'cliff_delta': cliff_delta,
                'cliff_interpretation': cliff_interp
            })
        
        return effect_sizes
    
    def _interpret_cohens_h(self, h):
        """Interpret Cohen's h effect size."""
        h = abs(h)
        if h < 0.2:
            return "Small"
        elif h < 0.5:
            return "Medium"
        elif h < 0.8:
            return "Large"
        else:
            return "Very Large"
    
    def _interpret_cliff_delta(self, d):
        """Interpret Cliff's Delta."""
        d = abs(d)
        if d < 0.147:
            return "Negligible"
        elif d < 0.33:
            return "Small"
        elif d < 0.474:
            return "Medium"
        else:
            return "Large"
    
    def metric_level_analysis(self):
        """
        Analyze performance at the metric level.
        """
        metric_performance = []
        
        for metric in self.binary_metrics:
            evalion_correct = (self.evalion_df[metric] == self.ground_truth_df[metric]).mean()
            cekura_correct = (self.cekura_df[metric] == self.ground_truth_df[metric]).mean()
            coval_correct = (self.coval_df[metric] == self.ground_truth_df[metric]).mean()
            
            # Chi-square test for this specific metric
            observed = np.array([
                [evalion_correct * self.n_recordings, (1-evalion_correct) * self.n_recordings],
                [cekura_correct * self.n_recordings, (1-cekura_correct) * self.n_recordings],
                [coval_correct * self.n_recordings, (1-coval_correct) * self.n_recordings]
            ])
            chi2, p, dof, expected = chi2_contingency(observed)
            
            metric_performance.append({
                'metric': metric,
                'evalion_accuracy': evalion_correct,
                'cekura_accuracy': cekura_correct,
                'coval_accuracy': coval_correct,
                'chi2_statistic': chi2,
                'p_value': p,
                'max_diff': max(evalion_correct, cekura_correct, coval_correct) - 
                           min(evalion_correct, cekura_correct, coval_correct)
            })
        
        return pd.DataFrame(metric_performance)
    
    def run_complete_analysis(self):
        """
        Run all statistical tests and return comprehensive results.
        """
        print("="*70)
        print("COMPREHENSIVE STATISTICAL ANALYSIS OF PLATFORM PERFORMANCE")
        print("="*70)
        print(f"Data: {self.n_recordings} recordings × {self.n_binary_metrics} metrics = "
              f"{self.n_recordings * self.n_binary_metrics} paired observations per platform")
        print()
        
        # 1. Omnibus test
        print("1. OMNIBUS TEST (Cochran's Q)")
        print("-"*50)
        cochrans = self.cochrans_q_test()
        print(f"Q-statistic: {cochrans['statistic']:.3f}")
        print(f"p-value: {cochrans['p_value']:.6f}")
        print(f"Interpretation: {'Significant differences exist' if cochrans['p_value'] < 0.05 else 'No significant differences'}")
        print()
        
        # 2. Pairwise comparisons
        print("2. PAIRWISE COMPARISONS (McNemar's Test)")
        print("-"*50)
        mcnemar_results = self.mcnemar_pairwise_tests()
        for result in mcnemar_results:
            print(f"\n{result['comparison']}:")
            print(f"  Test: {result['test_type']}")
            print(f"  Statistic: {result['statistic']:.3f}")
            print(f"  p-value (raw): {result['p_value']:.6f}")
            print(f"  p-value (Bonferroni): {result['p_adjusted']:.6f}")
            print(f"  Significant: {'Yes' if result['significant'] else 'No'}")
            print(f"  Agreement: {result['agreement']:.3f}")
            print(f"  Cohen's κ: {result['cohens_kappa']:.3f}")
            cont = result['contingency_table']
            print(f"  Contingency: Both correct={cont['both_correct']}, "
                  f"Discordant={cont[list(cont.keys())[1]] + cont[list(cont.keys())[2]]}")
        
        # 3. Effect sizes
        print("\n3. EFFECT SIZES")
        print("-"*50)
        effect_sizes = self.calculate_effect_sizes()
        for effect in effect_sizes:
            print(f"\n{effect['comparison']}:")
            print(f"  Proportion difference: {effect['prop_diff']:.3f}")
            print(f"  Cohen's h: {effect['cohens_h']:.3f} ({effect['h_interpretation']})")
            print(f"  Cliff's Delta: {effect['cliff_delta']:.3f} ({effect['cliff_interpretation']})")
        
        # 4. Metric-level analysis
        print("\n4. METRIC-LEVEL PERFORMANCE")
        print("-"*50)
        metric_df = self.metric_level_analysis()
        print(metric_df.to_string())
        
        return {
            'cochrans_q': cochrans,
            'mcnemar': mcnemar_results,
            'effect_sizes': effect_sizes,
            'metric_performance': metric_df
        }


# Usage with your actual data
def perform_statistical_analysis(evalion_df, cekura_df, coval_df, ground_truth_df,
                                binary_metric_names, numeric_metric_names):
    """
    Main function to run statistical analysis on your evaluation data.
    """
    analyzer = EvaluationStatisticalAnalysis(
        evalion_df=evalion_df,
        cekura_df=cekura_df,
        coval_df=coval_df,
        ground_truth_df=ground_truth_df,
        binary_metrics=binary_metric_names,
        numeric_metrics=numeric_metric_names
    )
    
    results = analyzer.run_complete_analysis()
    
    # Create summary for paper
    print("\n" + "="*70)
    print("SUMMARY FOR PAPER")
    print("="*70)
    
    # Overall accuracies
    binary_outcomes = analyzer.prepare_binary_outcomes()
    print("\nOverall Accuracy:")
    for platform, outcomes in binary_outcomes.items():
        print(f"  {platform}: {np.mean(outcomes):.3f}")
    
    print("\nStatistical Significance:")
    print(f"  Cochran's Q(2) = {results['cochrans_q']['statistic']:.2f}, "
          f"p {'< 0.001' if results['cochrans_q']['p_value'] < 0.001 else '= ' + format(results['cochrans_q']['p_value'], '.3f')}")
    
    print("\nPairwise Comparisons (McNemar's test with Bonferroni correction):")
    for mc in results['mcnemar']:
        sig_marker = "***" if mc['p_adjusted'] < 0.001 else "**" if mc['p_adjusted'] < 0.01 else "*" if mc['p_adjusted'] < 0.05 else "ns"
        print(f"  {mc['comparison']}: stat = {mc['statistic']:.3f}, p = {mc['p_adjusted']:.3f} {sig_marker}, κ = {mc['cohens_kappa']:.3f}")
    
    return results


In [ ]:
# Run the comprehensive statistical analysis
print("🔬 RUNNING COMPREHENSIVE STATISTICAL ANALYSIS")
print("="*60)

results = perform_statistical_analysis(
    evalion_results_df, 
    cekura_results_df,
    coval_results_df,
    ground_truth_df,
    binary_metric_names,
    numeric_metric_names
)

print("\n" + "🎯 ANALYSIS COMPLETE!")
print("="*60)
print("Results stored in 'results' variable with the following keys:")
for key in results.keys():
    print(f"  - {key}")

print(f"\n📊 Quick Summary:")
print(f"Cochran's Q p-value: {results['cochrans_q']['p_value']:.6f}")
print(f"Number of pairwise comparisons: {len(results['mcnemar'])}")
print(f"Significant pairwise comparisons: {sum(1 for r in results['mcnemar'] if r['significant'])}")

# Show metric-level performance table
print(f"\n📈 Metric-Level Performance:")
print(results['metric_performance'].round(3))

## STATISTICAL ANALYSIS - BASED ON FSCORE

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import bootstrap
import warnings
warnings.filterwarnings('ignore')

class F1ScoreStatisticalAnalysis:
    """
    Statistical analysis based on F1-scores for comparing voice AI evaluation platforms.
    Uses bootstrap methods and permutation tests appropriate for F1-score distributions.
    """
    
    def __init__(self, 
                 evalion_scores_df: pd.DataFrame,
                 cekura_scores_df: pd.DataFrame,
                 coval_scores_df: pd.DataFrame,
                 evalion_df: pd.DataFrame,
                 cekura_df: pd.DataFrame,
                 coval_df: pd.DataFrame,
                 ground_truth_df: pd.DataFrame,
                 binary_metrics: list[str]):
        """
        Initialize with both score DataFrames and raw evaluation DataFrames.
        """
        self.evalion_scores = evalion_scores_df
        self.cekura_scores = cekura_scores_df
        self.coval_scores = coval_scores_df
        
        self.evalion_df = evalion_df
        self.cekura_df = cekura_df
        self.coval_df = coval_df
        self.ground_truth_df = ground_truth_df
        self.binary_metrics = binary_metrics
        
        # Ensure alignment
        common_idx = (evalion_df.index.intersection(cekura_df.index)
                     .intersection(coval_df.index)
                     .intersection(ground_truth_df.index))
        
        self.n_recordings = len(common_idx)
        self.n_metrics = len(binary_metrics)
        
    def bootstrap_confidence_intervals(self, n_bootstrap=10000):
        """
        Calculate bootstrap confidence intervals for F1-scores.
        """
        results = {}
        
        for platform_name, scores_df in [
            ('Evalion', self.evalion_scores),
            ('Cekura', self.cekura_scores),
            ('Coval', self.coval_scores)
        ]:
            f1_scores = scores_df['f1_score'].values
            
            # Bootstrap for mean F1-score
            bootstrap_means = []
            for _ in range(n_bootstrap):
                sample = np.random.choice(f1_scores, size=len(f1_scores), replace=True)
                bootstrap_means.append(np.mean(sample))
            
            # Calculate 95% CI
            ci_lower = np.percentile(bootstrap_means, 2.5)
            ci_upper = np.percentile(bootstrap_means, 97.5)
            
            results[platform_name] = {
                'mean_f1': np.mean(f1_scores),
                'std_f1': np.std(f1_scores),
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'bootstrap_distribution': bootstrap_means
            }
        
        return results
    
    def permutation_test_f1_scores(self, platform1_scores, platform2_scores, n_permutations=10000):
        """
        Perform permutation test for difference in F1-scores.
        """
        # Observed difference
        observed_diff = np.mean(platform1_scores) - np.mean(platform2_scores)
        
        # Combine scores for permutation
        combined = np.concatenate([platform1_scores, platform2_scores])
        n1 = len(platform1_scores)
        
        # Permutation test
        perm_diffs = []
        for _ in range(n_permutations):
            np.random.shuffle(combined)
            perm_p1 = combined[:n1]
            perm_p2 = combined[n1:]
            perm_diff = np.mean(perm_p1) - np.mean(perm_p2)
            perm_diffs.append(perm_diff)
        
        # Calculate p-value
        p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))
        
        return {
            'observed_difference': observed_diff,
            'p_value': p_value,
            'permutation_distribution': perm_diffs
        }
    
    def recording_level_bootstrap(self, n_bootstrap=10000):
        """
        Bootstrap F1-scores at the recording level to account for dependencies.
        """
        results = {}
        
        for platform_name, platform_df in [
            ('Evalion', self.evalion_df),
            ('Cekura', self.cekura_df),
            ('Coval', self.coval_df)
        ]:
            bootstrap_f1s = []
            recording_ids = list(self.ground_truth_df.index)
            
            for _ in range(n_bootstrap):
                # Sample recordings with replacement
                sampled_recordings = np.random.choice(recording_ids, 
                                                     size=len(recording_ids), 
                                                     replace=True)
                
                # Calculate F1 for this bootstrap sample
                tp = fp = fn = tn = 0
                for metric in self.binary_metrics:
                    for rec_id in sampled_recordings:
                        pred = platform_df.loc[rec_id, metric]
                        truth = self.ground_truth_df.loc[rec_id, metric]
                        
                        if pred == 1 and truth == 1:
                            tp += 1
                        elif pred == 1 and truth == 0:
                            fp += 1
                        elif pred == 0 and truth == 1:
                            fn += 1
                        else:
                            tn += 1
                
                # Calculate F1 for this bootstrap sample
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                bootstrap_f1s.append(f1)
            
            results[platform_name] = {
                'bootstrap_f1_scores': bootstrap_f1s,
                'mean_f1': np.mean(bootstrap_f1s),
                'ci_lower': np.percentile(bootstrap_f1s, 2.5),
                'ci_upper': np.percentile(bootstrap_f1s, 97.5)
            }
        
        return results
    
    def pairwise_f1_comparisons(self):
        """
        Perform all pairwise comparisons using permutation tests.
        """
        comparisons = []
        
        platform_scores = {
            'Evalion': self.evalion_scores['f1_score'].values,
            'Cekura': self.cekura_scores['f1_score'].values,
            'Coval': self.coval_scores['f1_score'].values
        }
        
        pairs = [
            ('Evalion', 'Cekura'),
            ('Evalion', 'Coval'),
            ('Cekura', 'Coval')
        ]
        
        for p1, p2 in pairs:
            result = self.permutation_test_f1_scores(
                platform_scores[p1],
                platform_scores[p2]
            )
            
            # Calculate relative improvement
            mean_p1 = np.mean(platform_scores[p1])
            mean_p2 = np.mean(platform_scores[p2])
            relative_improvement = (mean_p1 - mean_p2) / mean_p2 if mean_p2 > 0 else np.inf
            
            comparisons.append({
                'comparison': f'{p1} vs {p2}',
                'p1_mean_f1': mean_p1,
                'p2_mean_f1': mean_p2,
                'f1_difference': result['observed_difference'],
                'p_value': result['p_value'],
                'relative_improvement': relative_improvement,
                'practical_impact_per_1000': int(result['observed_difference'] * 1000)
            })
        
        return comparisons
    
    def metric_specific_analysis(self):
        """
        Analyze F1-scores and confidence intervals for each metric.
        """
        metric_results = []
        
        for i, metric in enumerate(self.binary_metrics):
            metric_data = {
                'metric': metric,
                'evalion_f1': self.evalion_scores.iloc[i]['f1_score'],
                'cekura_f1': self.cekura_scores.iloc[i]['f1_score'],
                'coval_f1': self.coval_scores.iloc[i]['f1_score']
            }
            
            # Bootstrap CIs for this specific metric
            for platform_name, platform_df in [
                ('evalion', self.evalion_df),
                ('cekura', self.cekura_df),
                ('coval', self.coval_df)
            ]:
                bootstrap_f1s = []
                recording_ids = list(self.ground_truth_df.index)
                
                for _ in range(1000):  # Fewer iterations for individual metrics
                    sampled_recs = np.random.choice(recording_ids, 
                                                   size=len(recording_ids), 
                                                   replace=True)
                    
                    tp = fp = fn = 0
                    for rec_id in sampled_recs:
                        pred = platform_df.loc[rec_id, metric]
                        truth = self.ground_truth_df.loc[rec_id, metric]
                        
                        if pred == 1 and truth == 1:
                            tp += 1
                        elif pred == 1 and truth == 0:
                            fp += 1
                        elif pred == 0 and truth == 1:
                            fn += 1
                    
                    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                    bootstrap_f1s.append(f1)
                
                metric_data[f'{platform_name}_ci_lower'] = np.percentile(bootstrap_f1s, 2.5)
                metric_data[f'{platform_name}_ci_upper'] = np.percentile(bootstrap_f1s, 97.5)
            
            metric_results.append(metric_data)
        
        return pd.DataFrame(metric_results)
    
    def calculate_consistency(self):
        """
        Calculate coefficient of variation for each platform.
        """
        consistency = {}
        
        for platform_name, scores_df in [
            ('Evalion', self.evalion_scores),
            ('Cekura', self.cekura_scores),
            ('Coval', self.coval_scores)
        ]:
            f1_scores = scores_df['f1_score'].values
            mean_f1 = np.mean(f1_scores)
            std_f1 = np.std(f1_scores)
            cv = (std_f1 / mean_f1) * 100 if mean_f1 > 0 else np.inf
            
            consistency[platform_name] = {
                'mean_f1': mean_f1,
                'std_f1': std_f1,
                'coefficient_of_variation': cv,
                'f1_range': f1_scores.max() - f1_scores.min(),
                'min_f1': f1_scores.min(),
                'max_f1': f1_scores.max()
            }
        
        return consistency
    
    def run_complete_f1_analysis(self):
        """
        Run the complete F1-score based statistical analysis.
        """
        print("="*70)
        print("F1-SCORE BASED STATISTICAL ANALYSIS")
        print("="*70)
        print(f"Data: {self.n_recordings} recordings × {self.n_metrics} metrics")
        print()
        
        # 1. Bootstrap confidence intervals
        print("1. BOOTSTRAP CONFIDENCE INTERVALS (10,000 iterations)")
        print("-"*50)
        ci_results = self.bootstrap_confidence_intervals()
        
        print("| Platform | Mean F1-Score | 95% Bootstrap CI |")
        print("|----------|--------------|------------------|")
        for platform, results in ci_results.items():
            print(f"| {platform:8} | {results['mean_f1']:.3f} | [{results['ci_lower']:.3f}, {results['ci_upper']:.3f}] |")
        print()
        
        # 2. Pairwise comparisons
        print("2. PAIRWISE F1-SCORE COMPARISONS (Permutation Tests)")
        print("-"*50)
        comparisons = self.pairwise_f1_comparisons()
        
        print("| Comparison | F1 Diff | p-value | Relative Improvement | Per 1000 Calls |")
        print("|------------|---------|---------|---------------------|----------------|")
        for comp in comparisons:
            sig = "***" if comp['p_value'] < 0.001 else "**" if comp['p_value'] < 0.01 else "*" if comp['p_value'] < 0.05 else ""
            print(f"| {comp['comparison']:18} | {comp['f1_difference']:.3f} | "
                  f"{comp['p_value']:.4f}{sig:3} | {comp['relative_improvement']*100:18.1f}% | "
                  f"{comp['practical_impact_per_1000']:14} |")
        print()
        
        # 3. Platform consistency
        print("3. PLATFORM CONSISTENCY ANALYSIS")
        print("-"*50)
        consistency = self.calculate_consistency()
        
        print("| Platform | Mean F1 | Std Dev | CV (%) | Range | Min-Max |")
        print("|----------|---------|---------|--------|-------|---------|")
        for platform, metrics in consistency.items():
            print(f"| {platform:8} | {metrics['mean_f1']:.3f} | {metrics['std_f1']:.3f} | "
                  f"{metrics['coefficient_of_variation']:6.1f} | {metrics['f1_range']:.3f} | "
                  f"{metrics['min_f1']:.3f}-{metrics['max_f1']:.3f} |")
        print()
        
        # 4. Metric-specific analysis
        print("4. METRIC-SPECIFIC F1-SCORES")
        print("-"*50)
        metric_df = self.metric_specific_analysis()
        print(metric_df[['metric', 'evalion_f1', 'cekura_f1', 'coval_f1']].to_string(index=False))
        
        return {
            'confidence_intervals': ci_results,
            'pairwise_comparisons': comparisons,
            'consistency': consistency,
            'metric_analysis': metric_df
        }


# Function to use with your notebook data
def perform_f1_statistical_analysis(evalion_binary_scores_df, cekura_binary_scores_df, 
                                   coval_binary_scores_df, evalion_results_df, 
                                   cekura_results_df, coval_results_df, 
                                   ground_truth_df, binary_metric_names):
    """
    Main function to run F1-score based statistical analysis.
    """
    print("🔬 INITIALIZING F1-SCORE STATISTICAL ANALYSIS")
    print("="*60)
    
    analyzer = F1ScoreStatisticalAnalysis(
        evalion_scores_df=evalion_binary_scores_df,
        cekura_scores_df=cekura_binary_scores_df,
        coval_scores_df=coval_binary_scores_df,
        evalion_df=evalion_results_df,
        cekura_df=cekura_results_df,
        coval_df=coval_results_df,
        ground_truth_df=ground_truth_df,
        binary_metrics=binary_metric_names
    )
    
    results = analyzer.run_complete_f1_analysis()
    
    # Generate summary for paper
    print("\n" + "="*70)
    print("SUMMARY FOR PAPER")
    print("="*70)
    
    # Key findings
    print("\n📊 KEY FINDINGS:")
    print("-"*40)
    
    # Overall performance
    ci_results = results['confidence_intervals']
    print("\n1. OVERALL F1-SCORE PERFORMANCE:")
    for platform in ['Evalion', 'Cekura', 'Coval']:
        ci = ci_results[platform]
        print(f"   {platform}: {ci['mean_f1']:.3f} (95% CI: [{ci['ci_lower']:.3f}, {ci['ci_upper']:.3f}])")
    
    # Statistical significance
    print("\n2. STATISTICAL SIGNIFICANCE (Permutation Tests):")
    for comp in results['pairwise_comparisons']:
        if comp['p_value'] < 0.001:
            sig = "p < 0.001***"
        else:
            sig = f"p = {comp['p_value']:.3f}"
        print(f"   {comp['comparison']}: {sig}, Δ = {comp['f1_difference']:.3f}")
    
    # Practical significance
    print("\n3. PRACTICAL SIGNIFICANCE:")
    best_comp = max(results['pairwise_comparisons'], 
                    key=lambda x: abs(x['f1_difference']))
    print(f"   Largest gap: {best_comp['comparison']}")
    print(f"   - F1 difference: {best_comp['f1_difference']:.3f}")
    print(f"   - Relative improvement: {best_comp['relative_improvement']*100:.1f}%")
    print(f"   - Impact: {best_comp['practical_impact_per_1000']} additional correct per 1000 evaluations")
    
    # Consistency
    print("\n4. PERFORMANCE CONSISTENCY:")
    for platform, metrics in results['consistency'].items():
        print(f"   {platform}: CV = {metrics['coefficient_of_variation']:.1f}% (Range: {metrics['min_f1']:.3f}-{metrics['max_f1']:.3f})")
    
    print("\n" + "="*70)
    
    return results



In [ ]:
# Run the F1-score based statistical analysis
print("🔬 RUNNING F1-SCORE BASED STATISTICAL ANALYSIS")
print("="*60)

results_f1 = perform_f1_statistical_analysis(
    evalion_binary_scores_df, 
    cekura_binary_scores_df, 
    coval_binary_scores_df,
    evalion_results_df, 
    cekura_results_df, 
    coval_results_df,
    ground_truth_df, 
    binary_metric_names
)

# Save results for paper
f1_summary = {
    'confidence_intervals': results_f1['confidence_intervals'],
    'pairwise_comparisons': results_f1['pairwise_comparisons'],
    'consistency': results_f1['consistency'],
    'metric_analysis': results_f1['metric_analysis']
}

print("\n📈 Results saved in 'results_f1' variable")
print("Ready to use for paper's Statistical Analysis section")